# Lab: Getting Data from an API and a Web Page

Today we are going to learn how to fetch data from a web API and how to scrape a small HTML page and save as csv file.

| Part | Data comes from | You will end up with |
|---|---|---|
| A | Fake Store API (JSON) | products_df |
| B | A real web page (HTML) | countries_df and hockey_df |


Import the libraries

| Library | Used for |
|---|---|
| requests | Downloading data from a URL |
| pandas | DataFrames |
| BeautifulSoup (from bs4) | Searching inside HTML |



In [2]:
# write all your imports here
import requests
import pandas as pd
from bs4 import BeautifulSoup

## Part A: Data from a Web API

We use the **Fake Store API**, a free practice API with 20 shop products. Open this link in your browser first to see what the data looks like:

https://fakestoreapi.com/products

Remember the workflow from the lecture:

**URL → requests.get() → response → response.json() → pd.DataFrame()**

| Status code | Meaning |
|---|---|
| 200 | Success |
| 404 | Not found (wrong URL) |
| 500 | Problem on the server |

### Task 1: Send the request

Store the URL in a variable, send a GET request, and check that it worked.

You should see: Status code: 200

In [7]:
# Store "https://fakestoreapi.com/products" in a variable called api_url
api_url = "https://fakestoreapi.com/products"
# Send a GET request
response=requests.get(api_url,timeout=10)
# Print status code
print(response.status_code)

200


### Task 2: Look at the JSON before making a DataFrame

response.json() turns the JSON text into normal Python lists and dictionaries.

Always look at the data first: Is it a list or a dictionary? How many items? What does one item look like?

You should see: <class 'list'>, 20, and one product printed as a dictionary.

In [13]:
# store received json into data variable
data=response.json()
# check is it a list or a dict?
print(type(data))
# check how many products are there?
print(len(data))
# list the first product
print(data[0])
# print title and price of first product
print(data[0]['title'])
print(data[0]['price'])
# print rating of the first product
print(data[0]['rating'])


<class 'list'>
20
{'id': 1, 'title': 'Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops', 'price': 109.95, 'description': 'Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday', 'category': "men's clothing", 'image': 'https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png', 'rating': {'rate': 3.9, 'count': 120}}
Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops
109.95
{'rate': 3.9, 'count': 120}


### Task 3: Convert the JSON into a DataFrame

data is a **list of dictionaries**. Each dictionary becomes one row, and each key becomes a column.

You should see: shape (20, 7).

In [19]:
# convert data into data frame with name products
products_df=pd.DataFrame(data)
# display the head
display(products_df.head())
# display the shape
print(products_df.shape)
# display the columns
print(products_df.columns)
# display the products info
products_df.info()
# disply the products description
products_df.describe()

,id,title,price,description,category,image,rating
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,Your perfect pack for everyday use and walks i...,men's clothing,https://fakestoreapi.com/img/81fPKd-2AYL._AC_S...,"{'rate': 3.9, 'count': 120}"
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,"Slim-fitting style, contrast raglan long sleev...",men's clothing,https://fakestoreapi.com/img/71-3HjGNDUL._AC_S...,"{'rate': 4.1, 'count': 259}"
2,3,Mens Cotton Jacket,55.99,great outerwear jackets for Spring/Autumn/Wint...,men's clothing,https://fakestoreapi.com/img/71li-ujtlUL._AC_U...,"{'rate': 4.7, 'count': 500}"
3,4,Mens Casual Slim Fit,15.99,The color could be slightly different between ...,men's clothing,https://fakestoreapi.com/img/71YXzeOuslL._AC_U...,"{'rate': 2.1, 'count': 430}"
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,"From our Legends Collection, the Naga was insp...",jewelery,https://fakestoreapi.com/img/71pWzhdJNwL._AC_U...,"{'rate': 4.6, 'count': 400}"


(20, 7)
Index(['id', 'title', 'price', 'description', 'category', 'image', 'rating'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           20 non-null     int64  
 1   title        20 non-null     object 
 2   price        20 non-null     float64
 3   description  20 non-null     object 
 4   category     20 non-null     object 
 5   image        20 non-null     object 
 6   rating       20 non-null     object 
dtypes: float64(1), int64(1), object(5)
memory usage: 1.2+ KB


,id,price
count,20.00000,20.000000
mean,10.50000,162.046000
std,5.91608,272.220532
min,1.00000,7.950000
25%,5.75000,15.240000
50%,10.50000,56.490000
75%,15.25000,110.962500
max,20.00000,999.990000


### Task 4: Flatten the nested rating column

A cell holding a dictionary is hard to use. You cannot easily find the average rating, for example.

pd.json_normalize(data) works like pd.DataFrame(data), but it also splits inner dictionaries into separate columns. rating becomes two columns, rating.rate and rating.count.

You should see: a DataFrame with columns id, title, price, category, rate, count.

In [23]:
# convert rate dictionary into separate columns
products_df=pd.json_normalize(data)
# now print obtained columns
print(products_df.columns)
print(products_df.head())
# Keep only these columns: id, title, price, category, rating.rate, rating.count and store the result in products_df
products_df=products_df[['id','title','price','category','rating.rate','rating.count']]
# Rename "rating.rate" to "rate" and "rating.count" to "count"
products_df=products_df.rename(columns={'rating.rate':'rate','rating.count':'count'})
# Display products_df.head()
display(products_df.head())


Index(['id', 'title', 'price', 'description', 'category', 'image',
       'rating.rate', 'rating.count'],
      dtype='object')
   id                                              title   price  \
0   1  Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...  109.95   
1   2             Mens Casual Premium Slim Fit T-Shirts    22.30   
2   3                                 Mens Cotton Jacket   55.99   
3   4                               Mens Casual Slim Fit   15.99   
4   5  John Hardy Women's Legends Naga Gold & Silver ...  695.00   

                                         description        category  \
0  Your perfect pack for everyday use and walks i...  men's clothing   
1  Slim-fitting style, contrast raglan long sleev...  men's clothing   
2  great outerwear jackets for Spring/Autumn/Wint...  men's clothing   
3  The color could be slightly different between ...  men's clothing   
4  From our Legends Collection, the Naga was insp...        jewelery   

                              

,id,title,price,category,rate,count
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 ...",109.95,men's clothing,3.9,120
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,men's clothing,4.1,259
2,3,Mens Cotton Jacket,55.99,men's clothing,4.7,500
3,4,Mens Casual Slim Fit,15.99,men's clothing,2.1,430
4,5,John Hardy Women's Legends Naga Gold & Silver ...,695.00,jewelery,4.6,400


### Task 5: Save the products data to csv

In [24]:
# Save products_df to "products.csv" with index=False
products_df.to_csv('products.csv',index=False)

## Part B: Data from a Web Page (Scraping)

Now we download a **real page** from the internet. The steps after downloading are the same:
### Task 1: Shortcut for pages that have a table, pd.read_html()

If the data on a page is inside an HTML <table> pandas can read it in **one line**, with no BeautifulSoup needed.

pd.read_html() returns a **list** of DataFrames, one for each table on the page. Usually you want the first one, [0].

Open this page in your browser. It shows a table of hockey teams:
https://www.scrapethissite.com/pages/forms/

Note: read_html works only for real <table> tags. The countries page was not a table, which is why we needed BeautifulSoup there. We wrap the text in StringIO() because newer pandas versions expect that.

You should see: 1 table found, with columns like Team Name, Year, Wins, Losses.

In [30]:
# Request "https://www.scrapethissite.com/pages/forms/"
response=requests.get("https://www.scrapethissite.com/pages/forms/",timeout=10)
# read html and store in tables variable
tables=pd.read_html(response.text)
# display how many tables were found
print(len(tables))
# store table into hockey_df
hockey_df=tables[0]
# Display head, shape and data types
display(hockey_df.head())
print(hockey_df.shape)
print(hockey_df.dtypes)
# Display the 5 rows with the most Wins
hockey_df.sort_values(by='Wins',ascending=False).head()


1


/tmp/ipykernel_1038/1056833114.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables=pd.read_html(response.text)


,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278,14
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,NaN,0.425,273,298,-25


(25, 9)
Team Name              object
Year                    int64
Wins                    int64
Losses                  int64
OT Losses             float64
Win %                 float64
Goals For (GF)          int64
Goals Against (GA)      int64
+ / -                   int64
dtype: object


,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211,73
16,St. Louis Blues,1990,47,22,NaN,0.588,310,250,60
7,Los Angeles Kings,1990,46,24,NaN,0.575,340,254,86
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35


## Scraping using Beautiful Soup

**URL → requests.get() → response.text (HTML) → BeautifulSoup → find() / find_all() → list of dicts → DataFrame**

We use **Scrape This Site**, which was built for students to practise scraping. Open it in your browser:

https://www.scrapethissite.com/pages/simple/

It lists 250 countries. Right-click any country name and choose **Inspect**. You will see that every country is written like this:

```html
<div class="col-md-4 country">
    <h3 class="country-name"> Andorra </h3>
    <div class="country-info">
        <strong>Capital:</strong> <span class="country-capital">Andorra la Vella</span>
        <strong>Population:</strong> <span class="country-population">84000</span>
        <strong>Area (km2):</strong> <span class="country-area">468.0</span>
    </div>
</div>
```

Every value has its own class name, so it is easy to find:

| Value | Tag | Class |
|---|---|---|
| one whole country | div | country |
| name | h3 | country-name |
| capital | span | country-capital |
| population | span | country-population |
| area | span | country-area |

Remember:
- find() returns the **first** match. find_all() returns a **list** of all matches.
- Write class_ with an underscore.
- .get_text(strip=True) gives the text without extra spaces.

### Task 2: Download the page and make the soup

You should see: Status code: 200 and a page title that starts with Countries of the World.

In [36]:
# Store "https://www.scrapethissite.com/pages/simple/" in page_url
response=requests.get("https://www.scrapethissite.com/pages/simple/")
# request the web page and print the status code
print(response.status_code)
# Print the first 300 characters of response.text
print(response.text[:300])
# parse it using beautiful soup
soup=BeautifulSoup(response.text,'html.parser')
# find title of the web page
# print(soup.title)
print(soup.find('title').get_text(strip=True))


200
<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping</title>
    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" />

    <meta name="viewport" 
Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping


### Task 3: Get ONE country first

Before writing a loop, get the first country right. Search **inside** that one country block by calling .find() on it.

You should see: Andorra, Andorra la Vella, 84000, 468.0

In [38]:
# find the first div block of the country and store in
first=soup.find('div',class_='country')
print(first)
# store country name in name
name=first.find('h3',class_='country-name').get_text(strip=True)
# store capital name in capital
capital=first.find('span',class_='country-capital').get_text(strip=True)
# store population
population=first.find('span',class_='country-population').get_text(strip=True)
# store area
area=first.find('span',class_='country-area').get_text(strip=True)
# print all four
print(name,capital,population,area)


<div class="col-md-4 country">
<h3 class="country-name">
<i class="flag-icon flag-icon-ad"></i>
                            Andorra
                        </h3>
<div class="country-info">
<strong>Capital:</strong> <span class="country-capital">Andorra la Vella</span><br/>
<strong>Population:</strong> <span class="country-population">84000</span><br/>
<strong>Area (km<sup>2</sup>):</strong> <span class="country-area">468.0</span><br/>
</div>
</div>
Andorra Andorra la Vella 84000 468.0


### Task 4: Get ALL countries into a DataFrame

Now repeat 3 for every country with a loop. This is the same pattern as the lecture's table example:

1. Make an empty list rows.
2. Loop over every country block and append one dictionary per country.
3. After the loop, pd.DataFrame(rows).

You should see: shape (250, 4).

In [41]:
# do the same step for all countries using find all method
countries=soup.find_all('div',class_='country')
print(countries)
# use iterator to get details of each country one by one
rows=[]
for country in countries:
    name=country.find('h3',class_='country-name').get_text(strip=True)
    capital=country.find('span',class_='country-capital').get_text(strip=True)
    population=country.find('span',class_='country-population').get_text(strip=True)
    area=country.find('span',class_='country-area').get_text(strip=True)
# convert into dataframe
    rows.append({'name':name,'capital':capital,'population':population,'area':area})
countries_df=pd.DataFrame(rows)
# display the head and shape
display(countries_df.head())
print(countries_df.shape)


[<div class="col-md-4 country">
<h3 class="country-name">
<i class="flag-icon flag-icon-ad"></i>
                            Andorra
                        </h3>
<div class="country-info">
<strong>Capital:</strong> <span class="country-capital">Andorra la Vella</span><br/>
<strong>Population:</strong> <span class="country-population">84000</span><br/>
<strong>Area (km<sup>2</sup>):</strong> <span class="country-area">468.0</span><br/>
</div>
</div>, <div class="col-md-4 country">
<h3 class="country-name">
<i class="flag-icon flag-icon-ae"></i>
                            United Arab Emirates
                        </h3>
<div class="country-info">
<strong>Capital:</strong> <span class="country-capital">Abu Dhabi</span><br/>
<strong>Population:</strong> <span class="country-population">4975593</span><br/>
<strong>Area (km<sup>2</sup>):</strong> <span class="country-area">82880.0</span><br/>
</div>
</div>, <div class="col-md-4 country">
<h3 class="country-name">
<i class="flag-icon flag

,name,capital,population,area
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


(250, 4)


### Task 5: Clean and analyse

Everything scraped from HTML is **text**, even numbers. Check dtypes: population and area will show as object (text). You cannot sort or average them correctly until you convert them.

Here the numbers have no symbols, so .astype() is enough.

After cleaning: population is int64, area is float64.

In [43]:
# Print countries datatype
print(countries_df.dtypes)
# Convert population to int
countries_df['population']=countries_df['population'].astype(int)
# Convert area to float the same way
countries_df['area']=countries_df['area'].astype(float)
# Print countries_df.dtypes again
print(countries_df)

name          object
capital       object
population    object
area          object
dtype: object
                     name           capital  population       area
0                 Andorra  Andorra la Vella       84000      468.0
1    United Arab Emirates         Abu Dhabi     4975593    82880.0
2             Afghanistan             Kabul    29121286   647500.0
3     Antigua and Barbuda        St. John's       86754      443.0
4                Anguilla        The Valley       13254      102.0
..                    ...               ...         ...        ...
245                 Yemen             Sanaa    23495361   527970.0
246               Mayotte         Mamoudzou      159042      374.0
247          South Africa          Pretoria    49000000  1219912.0
248                Zambia            Lusaka    13460305   752614.0
249              Zimbabwe            Harare    11651858   390580.0

[250 rows x 4 columns]


In [44]:
# Save countries_df to csv
countries_df.to_csv('countries.csv',index=False)